# Crop Recommendation System: short demo
**Rishi Choudhary, Anant Srivastav (Mentor: Dr. Jaswant Kumar), Manipal University Jaipur**

This notebook downloads two public **Kaggle** datasets, trains **Random Forest** models, and recommends:
1. the **crop** to grow from soil N, P, K, pH and the weather (temperature, humidity, rainfall), and
2. a **fertilizer** for that crop.

| Dataset | Kaggle link | Rows |
|---|---|---|
| Crop Recommendation | [atharvaingle/crop-recommendation-dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset) | 2,200 (22 crops) |
| Fertilizer Prediction | [gdabhishek/fertilizer-prediction](https://www.kaggle.com/datasets/gdabhishek/fertilizer-prediction) | 99 (7 fertilizers) |

**How to run:** `Runtime → Run all`. Then change the numbers in the form of Step 4 and run that cell again.
No Kaggle login is needed. If the download fails, the first cell asks you to upload the CSV file instead.

*Both are public benchmark datasets, so the results show how the method works; they are not field-tested farming advice.*

In [ ]:
# Step 0: download the two Kaggle datasets
!pip -q install -U kagglehub

import glob
import kagglehub
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

def kaggle_csv(handle):
    """Download a public Kaggle dataset and read its CSV file."""
    try:
        folder = kagglehub.dataset_download(handle)
        path = glob.glob(f"{folder}/*.csv")[0]
    except Exception as err:                    # no internet or Kaggle down: upload the CSV by hand
        print(f"Download of {handle} failed ({err}). Please upload its CSV file:")
        from google.colab import files
        path = list(files.upload())[0]
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()         # one column name ends with a space
    return df

crop_df = kaggle_csv("atharvaingle/crop-recommendation-dataset")
fert_df = kaggle_csv("gdabhishek/fertilizer-prediction")
print("Crop dataset:", crop_df.shape, "-", crop_df["label"].nunique(), "crops")
print("Fertilizer dataset:", fert_df.shape, "-", fert_df["Fertilizer Name"].nunique(), "fertilizers")
crop_df.head()

## Step 1: Crop model (Random Forest)
We keep 20% of the data (440 samples) aside as unseen test data, and also run 5-fold cross-validation on all 2,200 samples.

In [ ]:
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
X, y = crop_df[FEATURES], crop_df["label"]

# 80% to train, 20% kept aside for testing (20 samples of each crop)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
crop_model = RandomForestClassifier(n_estimators=100, random_state=42)
crop_model.fit(X_train, y_train)
print(f"Accuracy on {len(y_test)} unseen test samples: {crop_model.score(X_test, y_test):.2%}")

# 5-fold cross-validation: every sample is tested once
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X, y, cv=folds)
print(f"5-fold cross-validation accuracy: {cv.mean():.2%} ± {cv.std():.2%}")

## Step 2: Fertilizer advice for the recommended crop
**gap = what the crop needs − what the soil has**, for N, P and K. The largest gap decides:
if the soil is short, add that nutrient (urea for N, DAP or SSP for P, MOP for K);
if the soil already has more than the crop needs, skip it this season.
The crop needs come from the Harvestify requirement table, the same table as in our paper.

In [ ]:
# N, P, K each crop needs (Harvestify requirement table)
NEEDS = {
    "apple": (20, 125, 200), "banana": (100, 75, 50), "blackgram": (40, 60, 20), "chickpea": (40, 60, 80),
    "coconut": (20, 10, 30), "coffee": (100, 20, 30), "cotton": (120, 40, 20), "grapes": (20, 125, 200),
    "jute": (80, 40, 40), "kidneybeans": (20, 60, 20), "lentil": (20, 60, 20), "maize": (80, 40, 20),
    "mango": (20, 20, 30), "mothbeans": (20, 40, 20), "mungbean": (20, 40, 20), "muskmelon": (100, 10, 50),
    "orange": (20, 10, 10), "papaya": (50, 50, 50), "pigeonpeas": (20, 60, 20), "pomegranate": (20, 10, 40),
    "rice": (80, 40, 40), "watermelon": (100, 10, 50),
}
ADVICE = {("N", "low"): "add nitrogen, e.g. urea",          ("N", "high"): "skip nitrogen this season",
          ("P", "low"): "add phosphate, e.g. DAP or SSP",   ("P", "high"): "skip phosphate this season",
          ("K", "low"): "add potash, e.g. MOP",             ("K", "high"): "skip potash this season"}

def fertilizer_advice(crop, N, P, K):
    gaps = dict(zip("NPK", np.subtract(NEEDS[crop], (N, P, K))))
    lines = [f"  {n}: soil has {s:g}, {crop} needs {need:g} (gap {gaps[n]:+g})"
             for n, s, need in zip("NPK", (N, P, K), NEEDS[crop])]
    nutrient = max(gaps, key=lambda n: abs(gaps[n]))     # the largest gap decides
    if gaps[nutrient] == 0:
        return "\n".join(lines + ["  -> soil N, P and K already match this crop"])
    status = "low" if gaps[nutrient] > 0 else "high"
    return "\n".join(lines + [f"  -> {nutrient} is {status} by {abs(gaps[nutrient]):g}: {ADVICE[(nutrient, status)]}"])

print(fertilizer_advice("rice", 90, 42, 43))

## Step 3: Fertilizer product model (Random Forest)
A second Random Forest learns the fertilizer name (Urea, DAP, 14-35-14, ...) from the Kaggle fertilizer dataset.
It has only 99 rows and covers a narrow range of soils, so Step 5 warns you when an input is outside that range.

In [ ]:
NUM = ["Temparature", "Humidity", "Moisture", "Nitrogen", "Phosphorous", "Potassium"]   # spelling as in the dataset
CAT = ["Soil Type", "Crop Type"]
Xf = pd.get_dummies(fert_df[NUM + CAT])        # soil type and crop type become 0/1 columns
yf = fert_df["Fertilizer Name"]

fert_model = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xf, yf)
cv = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), Xf, yf, cv=folds)
print(f"Fertilizer model, 5-fold cross-validation accuracy: {cv.mean():.2%} ± {cv.std():.2%}")
print("Training range:", ", ".join(f"{c} {fert_df[c].min()}-{fert_df[c].max()}" for c in NUM))

## Step 4: Try it: which crop, and which fertilizer?
Change the soil test and weather values in the form, then run the cell.

In [ ]:
#@title Soil test and weather values
N = 90  #@param {type:"number"}
P = 42  #@param {type:"number"}
K = 43  #@param {type:"number"}
temperature = 20.9  #@param {type:"number"}
humidity = 82.0  #@param {type:"number"}
ph = 6.5  #@param {type:"number"}
rainfall = 202.9  #@param {type:"number"}

farmer = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]], columns=FEATURES)
proba = crop_model.predict_proba(farmer)[0]
top3 = np.argsort(proba)[::-1][:3]

print("Recommended crops:")
for rank, i in enumerate(top3, 1):
    print(f"  {rank}. {crop_model.classes_[i]:12s} probability {proba[i]:.2f}")

best = crop_model.classes_[top3[0]]
print(f"\nFertilizer advice for {best}:")
print(fertilizer_advice(best, N, P, K))

## Step 5 (optional): Predict the fertilizer product
Uses the fertilizer Random Forest from Step 3. The default values are the first row of the dataset.

In [ ]:
#@title Fertilizer model inputs
f_temperature = 26  #@param {type:"number"}
f_humidity = 52  #@param {type:"number"}
f_moisture = 38  #@param {type:"number"}
soil_type = "Sandy"  #@param ["Sandy", "Loamy", "Black", "Red", "Clayey"]
crop_type = "Maize"  #@param ["Maize", "Sugarcane", "Cotton", "Tobacco", "Paddy", "Barley", "Wheat", "Millets", "Oil seeds", "Pulses", "Ground Nuts"]
f_nitrogen = 37  #@param {type:"number"}
f_phosphorous = 0  #@param {type:"number"}
f_potassium = 0  #@param {type:"number"}

values = [f_temperature, f_humidity, f_moisture, f_nitrogen, f_phosphorous, f_potassium]
for col, v in zip(NUM, values):
    if not fert_df[col].min() <= v <= fert_df[col].max():
        print(f"Note: {col} = {v} is outside {fert_df[col].min()}-{fert_df[col].max()}, "
              "the range of the training data, so this prediction may not be reliable.")
row = pd.get_dummies(pd.DataFrame([values + [soil_type, crop_type]], columns=NUM + CAT))
row = row.reindex(columns=Xf.columns, fill_value=False)
print("Recommended fertilizer:", fert_model.predict(row)[0])